In [ ]:
%load_ext autoreload
%autoreload 2

# MCAP

In [ ]:
from pathlib import Path

from psilia import get_console
from psilia.data.mcap import (
    get_mcap_overview, 
    get_mcaps, 
    get_summary,
    get_topics,
    get_schemas,
    get_channel_overview,
    read_nth_message,
    parse_msg,
    parse_ros_msg,
)
from psilia.data.lance import convert
from pandas import DataFrame
from psilia.data.lance import convert
from psilia.data.utils import save_yaml, load_yaml, psi_glob
from mcap_ros2.reader import read_ros2_messages
import matplotlib.pyplot as plt
import numpy as np

console = get_console()
print = console.print

In [ ]:
path = str(Path('~/workspace/data/rosbags/**/*.mcap').expanduser())
mcaps = psi_glob(path)
console.print(mcaps[["name", "path"]])

mcap = mcaps["path"][1]
overview = get_mcap_overview(mcap)
console.print(overview)

In [ ]:
meta = load_yaml(mcap.parent / "metadata.yaml")
console.print(meta)
t0 = meta['rosbag2_bagfile_information']['starting_time']['nanoseconds_since_epoch']

In [ ]:
console.print(get_channel_overview(mcap))

In [ ]:
from psilia.data.mcap import McapTaker as Taker
from psilia.transforms import Transform


taker = Taker(mcap, 
    topic_map={"/zed/zed_node/pose":"/pose", "/zed/zed_node/left/camera_info": "/caminfo"}, 
    topic_transforms={"/pose": {
        "__node__": lambda d: Transform.from_dict(d)
    }})
keys = [
    "/pose", 
    "/pose:translation", 
    "/caminfo", 
    "/zed/zed_node/pose_with_covariance"
]

p, ts, info, pcs = taker.k[*keys].i[0,:5,0, 0].t(3., 4.)
# ======================================================
print("type(p)=type(p)")
print(f"type(ts)={type(ts)}, len(ts)={len(ts)}")
print(f"type(info)={type(info)}")
print(f"type(pcs)={type(pcs)}")

In [ ]:
taker = Taker(mcap)
keys = ["/tf:transforms", "/zed/zed_node/pose:translation"]
data = taker.k[*keys].i[0,:2].t(3., 4.)

for k,r in data.items():
    print(f"'{k}':", r)

In [ ]:
topic = "/pose"
keys = [f"{topic}:log_time", f"{topic}:publish_time", f"{topic}:message_time"]

data = taker.k[*keys].i[:,:,:].t(reverse=False)

# ==================================================
fig, axs = plt.subplots(3,1, figsize=(5,4.5))
fig.suptitle("Interarrival times")
for i, (k,v) in enumerate(data.items()):
    ts = np.array(v)*1e-9
    axs[i].set_yscale("log")
    axs[i].set_title(f"\"{k}\"")
    axs[i].hist(np.diff(ts), bins=100, color="r");
fig.tight_layout()
# ==================================================
fig, axs = plt.subplots(3,1, figsize=(5,4.5),)
fig.suptitle("Lags")
axs[0].set_title(f"\"{keys[1]}\" $-$ \"{keys[0]}\"")
axs[0].set_yscale("log")
axs[0].hist( np.array(data[1])*1e-9 - np.array(data[0])*1e-9, bins=100)
axs[1].set_title(f"\"{keys[2]}\" $-$ \"{keys[0]}\"")
axs[1].set_yscale("log")
axs[1].hist( np.array(data[2])*1e-9 - np.array(data[0])*1e-9, bins=100);
axs[2].set_title(f"\"{keys[2]}\" $-$ \"{keys[1]}\"")
axs[2].set_yscale("log")
axs[2].hist( np.array(data[2])*1e-9 - np.array(data[1])*1e-9, bins=100);
fig.tight_layout()